# Uitdaging: Tekstanalyse over Data Science

In dit voorbeeld doen we een eenvoudige oefening die alle stappen van een traditioneel data science-proces omvat. Je hoeft geen code te schrijven, je kunt gewoon op de cellen hieronder klikken om ze uit te voeren en het resultaat te bekijken. Als uitdaging wordt je aangemoedigd om deze code uit te proberen met andere data.

## Doel

In deze les hebben we verschillende concepten met betrekking tot Data Science besproken. Laten we proberen meer gerelateerde concepten te ontdekken door wat **text mining** te doen. We beginnen met een tekst over Data Science, halen er trefwoorden uit en proberen het resultaat vervolgens te visualiseren.

Als tekst gebruik ik de pagina over Data Science van Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Stap 1: De gegevens verkrijgen

De eerste stap in elk data science-proces is het verkrijgen van de gegevens. We zullen de `requests`-bibliotheek gebruiken om dat te doen:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Stap 2: De gegevens transformeren

De volgende stap is om de gegevens om te zetten in een voor verwerking geschikte vorm. In ons geval hebben we de HTML-broncode van de pagina gedownload en moeten we deze omzetten naar platte tekst.

Er zijn veel manieren om dit te doen. We zullen [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/) gebruiken, een populaire Python-bibliotheek voor het parsen van HTML. BeautifulSoup stelt ons in staat om specifieke HTML-elementen te targeten, zodat we ons kunnen concentreren op de hoofdinhoud van het artikel van Wikipedia en enkele navigatiemenu's, zijbalken, voetteksten en andere irrelevante inhoud kunnen verminderen (hoewel er nog steeds wat standaardtekst kan overblijven).


Eerst moeten we de BeautifulSoup-bibliotheek installeren voor het parsen van HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Stap 3: Inzichten verkrijgen

De belangrijkste stap is om onze data om te zetten in een vorm waaruit we inzichten kunnen halen. In ons geval willen we trefwoorden uit de tekst halen en zien welke trefwoorden betekenisvoller zijn.

We zullen de Python-bibliotheek genaamd [RAKE](https://github.com/aneesha/RAKE) gebruiken voor het extraheren van trefwoorden. Laten we eerst deze bibliotheek installeren indien deze nog niet aanwezig is:


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

De hoofdfunctie is beschikbaar via het `Rake`-object, dat we kunnen aanpassen met enkele parameters. In ons geval zullen we de minimale lengte van een trefwoord instellen op 5 tekens, de minimale frequentie van een trefwoord in het document op 3, en het maximale aantal woorden in een trefwoord op 2. Voel je vrij om met andere waarden te experimenteren en het resultaat te observeren.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


We hebben een lijst met termen verkregen samen met de bijbehorende mate van belang. Zoals je kunt zien, zijn de meest relevante vakgebieden, zoals machine learning en big data, aanwezig in de lijst op topposities.

## Stap 4: Het resultaat visualiseren

Mensen kunnen de gegevens het beste interpreteren in visuele vorm. Daarom is het vaak zinvol om de gegevens te visualiseren om enkele inzichten te verkrijgen. We kunnen de `matplotlib`-bibliotheek in Python gebruiken om een eenvoudige verdeling van de trefwoorden met hun relevantie te plotten:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Er is echter een nog betere manier om woordfrequenties te visualiseren - met behulp van **Word Cloud**. We moeten een andere bibliotheek installeren om de word cloud te plotten vanuit onze lijst met trefwoorden.


In [ ]:
!{sys.executable} -m pip install wordcloud

Het `WordCloud`-object is verantwoordelijk voor het verwerken van originele tekst of een vooraf berekende lijst met woorden en hun frequenties, en retourneert een afbeelding die vervolgens kan worden weergegeven met `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

We kunnen ook de originele tekst doorgeven aan `WordCloud` - laten we kijken of we een vergelijkbaar resultaat kunnen krijgen:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Je kunt zien dat de woordwolk er nu indrukwekkender uitziet, maar ook veel ruis bevat (bijv. niet-gerelateerde woorden zoals `Retrieved on`). Bovendien krijgen we minder trefwoorden die uit twee woorden bestaan, zoals *data scientist*, of *computer science*. Dit komt omdat het RAKE-algoritme veel beter is in het selecteren van goede trefwoorden uit tekst. Dit voorbeeld illustreert het belang van het voorbewerken en opschonen van data, omdat een duidelijk beeld aan het einde ons in staat stelt betere beslissingen te nemen.

In deze oefening hebben we een eenvoudig proces doorlopen om enige betekenis uit Wikipedia-tekst te halen, in de vorm van trefwoorden en een woordwolk. Dit voorbeeld is vrij eenvoudig, maar het toont goed alle typische stappen die een data scientist zal nemen bij het werken met data, vanaf het verzamelen van data tot aan visualisatie.

In onze cursus zullen we al deze stappen in detail bespreken.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
